In [1]:
!pip install -q -U transformers accelerate peft bitsandbytes

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B"
adapter_id = "zahidazam714/qwen-khowar-qlora-v2"

# Tokenizer load karo adapter repo se (kyunki v2 mein tokenizer bhi push kiya tha)
tokenizer = AutoTokenizer.from_pretrained(adapter_id)

# Base model load karo
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16,
    device_map="auto"
)

# Base model ka embedding resize karo agar tokenizer extend kiya tha
base_model.resize_token_embeddings(len(tokenizer))

# Ab LoRA adapter attach karo
model = PeftModel.from_pretrained(base_model, adapter_id)
model.eval()

print("Model + adapter loaded successfully")
print("Vocab size:", len(tokenizer))

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 35.2MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Model + adapter loaded successfully
Vocab size: 151665


In [3]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 39.6 MB/s eta 0:00:00


In [8]:
def generate(prompt, max_new_tokens=100, temperature=0.7):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False,
            top_p=0.9,
            repetition_penalty=1.15,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# Simple Khowar prompt test
test_prompt =" جہانگیر لالو ژاؤ اژیکو تھوویک پیڅھیسودیک کوری"
result = generate(test_prompt)
print(result)

 جہانگیر لالو ژاؤ اژیکو تھوویک پیڅھیسودیک کوریلیک۔ چاراہ دنیو سف مجبور بونی، ناموسوم اوشٹورئے رینجیتائے وا خوشانیو سورا کمپنیو وختہ کندوری روشتی شروع کورویان۔ ہر کما قسمہ کی ہسے کیاغ کی نو بویان ہموݰ ک


In [9]:
import zipfile
import os

zip_path = "/cpt-data-text.zip"  # apna actual zip filename yahan daalo
extract_to = "/content/khowar_data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

# Extracted files list karo taake final_file ka sahi path pata chale
for root, dirs, files in os.walk(extract_to):
    for f in files:
        print(os.path.join(root, f))

/content/khowar_data/cpt-data-text/02-Khowar Nama - Magazine - 11th Edition (Jan to Jun 2018) - 14905.txt
/content/khowar_data/cpt-data-text/07-Robinson Crusoe - Book - Translated by Farid Ahmad Raza - 9791.txt
/content/khowar_data/cpt-data-text/MIER collection all.txt
/content/khowar_data/cpt-data-text/01-Khowar Language Introduction - 1519.txt
/content/khowar_data/cpt-data-text/09-Afsans - Book - By Farid Ahmad Raza - 26556.txt
/content/khowar_data/cpt-data-text/08-First Bike Ride - Article - 492.txt
/content/khowar_data/cpt-data-text/03-Khowar Nama - Magazine - 12th Edition (Jan to Mar 2019) - 16500.txt
/content/khowar_data/cpt-data-text/10-Uray - Book - By Farid Ahmad Raza - 32258.txt
/content/khowar_data/cpt-data-text/12-Universal Declaration of Human Rights - Translated in Khowar by Farid Ahmad Raza - 4052.txt
/content/khowar_data/cpt-data-text/05-Qaso Kirdar - 166.txt
/content/khowar_data/cpt-data-text/06-Khowar Masaan Naam - 53.txt
/content/khowar_data/cpt-data-text/11-Fourth M

In [10]:
import math

# Apna raw Khowar text yahan paste karo (jo training mein nahi tha)
fresh_text = """
گرامبیشو سوم گرامبیشی

نیویشاک: صابر علی ریحان

قرآن پاکو سورا سفان ایمان شیر وا ہورو تعلیماتانتے لوڑیکو نان تتان سوم جام لو دیکو سوم جوستہ رشتہ دارانان، گرامبیشانان وا څھوغوان سوم شئیلی بیتی پیش بیکو اوچے جام لو دیکو تہکید کورونو بیتی شیر۔
جو اکھابیران چایو نشا گرم مشقولگیو موژی اسونی۔۔۔
لوؤ دونییان جو قسمہ زمانان۔۔
ای بوغدو ای ہنیسو۔۔۔ گیاک زمانہ ریکو رینیان کی ہتے وختہ اسپہ غون ویکوروم رویان التی اوغو درینی۔۔۔
اتالیق: ای زمانا گرامبیشو گرامبیشی اوشوئے، رشتہ خالدار جم رشتہ داری اوشوئے۔۔۔ تتو تت جوݰی څیق لوٹو درجہ بندی اوشوئے، کوس ماریک کورا کارا نو بریک دی اݰمارا واری اوشوئے۔
ݯارویلو: کیہ لوؤ دوس بوتہ کیاغ کوری ایغوتے انځئیک باؤاوتائے
گرامبیشو ژیری بیکو سوت بس پروشٹی دیک باؤاوتائے
 ایغو گنی کورمان کوریک، گوم لیہتہ گوم لیک باؤاوتائے
نہ سنی، نہ اسماعیلیہ دور دوری حالئیک باؤاوتائے
اللّہ اوّلی مه گرامبیشو تین دیار ریک باؤاوتائے
ای بیتی غمان پرویزی داڑئیک باؤاوتائے
څیڅیق ایغو گنی ڈول بیتی نست دریک باؤاوتائے
ایغو سوم مشقول کوری روشتیان خامئیک باؤاوتائے
نسین کی لاڑیسان بو خور کیہ اتالیق برار لاڑیسانہ؟؟؟۔۔۔
اتالیغ ران کی:
 ہنون زمانا ہر کوس تان ژانو عمل کیہ
کوس ݰپیک بو بائے تان شرانہ کانو عمل کیہ
خوشانیہ خفاگیہ کوس دی گیکو رائے نو بوئے
متین تان کی بوئے رے ہر کیاغ ہر کوس تان تھانو عمل کیہ
کوس بچین جام لوؤ نو ایغو شورئے دورین جنجال بو
سورانالی ته سوم مشقول چنگی ای جامو عمل کیہ
دوری څیقو کی لاڑیسان سے دی لوٹ اساقال
پروشٹی کوری دونیک نِکی فی الحال تان پھانو عمل کیہ
ریکو سوم صوبیدار تے موژی غیری ریتائے کی
ہنیسین تان ژانو بچ کوریلیک تو کیہ لوا صحیح دی کی استیاؤ پھیک بیتی حال بیلیک، کھیوتین کی ہنیسین ماریک اسقان بیتی شیر وا تو تان دورو رویان بچین بو شیرین، تان اژیلیان بچین پھیک بیتی حال بیلیک، اللّہ دی صبر کوراکان خوش داریران۔
لو ختم نو بوئے لو بو درونگار
اسپتین جانواران سار زیاد انسانانان سار خطرہ شیر،
اللّہ اسپہ سفاتین ایوال ایوالیو ہوݰ کوریکو توفیقو دیار ۔۔۔ انسان بیک اسقان انسانیت گران۔۔۔

پیشکش: #مئیرݯھترار

#صابر_علی_ریحان .             وختو پابندی (پھوپھوکان ادب)
             نیویشاک:فرید احمد رضا
شیر ترویو جماعتہ سبق ریران۔ چھوچھی ݱوت روپھی تان ہوستان وا موخو نیگی ہاسی بویان۔ ہاسی بیکار اچی تان سکولو بستو کاڑی ݯاکئے سکولوتین راہی کویان۔ شیر سکولہ توری تان ملگریانتین سلام کوروئے۔کلاسی استاذو لوان بو غورو سورا کار کوئے۔ استاذ کیہ ہدایات کی پرائے  استاذو ہدایاتان سورا پورا پورا عمل کوروئے۔
شیرو ای جم عادت ہیہ شیر کی ہیس بوتہ پوریکار پروشٹی تان کتابان،کاپیان وا قلم پنسلان لوڑی تان بستو لکھوئے۔ بستو شانہ ٹھونہ ݯاکئے لاکوئے۔ کوریکو کی ہو استاذ سکولہ کلاسی ای وختہ لو دیتی رے  اسیتائے کی کیہ سکولی څیق کی تان سکولو اشناریان صحیح حفاظت کوریکو کی نوبیتائے ہسے جم سبق ریکو نوبوئے وا وختہ سکولہ نو تارور۔
ہیغار پروشٹی شیر دی سکولہ توریکو ہمیشہ ہوتین گور باؤ اوتائے ،ہے بچین ہیس اسمبلیہ نو توراؤ اوشوئے۔ اسمبیلہ نو توریکو سکولہ ہو جرمانہ دی کوراؤ اوشونی۔ پروشٹی ہیس سکولار گیتی زاپان ای ژاغا، بستو خور ژاغہ وا کاپی اوچے قلمان کورا دی پیڅھاؤ اوشوئے وا چھوچھی سکولوتین راہی بیتی ہیتان مݰکی گراری باؤ اوتائے۔
استاذ کیہ وت کی وختو پابندیو بارا لو دیتی رے اسیتائے کی "وختو پابندی کیہ بولوٹ  دونیکو بݰ اشناری نو، بلکہ ہیس بو پھوک کورمہ شروع بوئے وا بو فائدہ تاریر۔" استاذ ہیہ لوؤ  دی  دیتی اسیتائے  کی وختو پابندی زندگیو  اسقان وا نظم و ضبطو پابند ساؤزئیر۔
استاذ وا ہموݰ دی رے اسیتائے کی پِیسہ مہ ای کما لوان سورا عمل کورور پِیسا تین اسقانی پیدا بوئے وا ہتیت ہمیت شینی۔  سکولار چھٹی بیتی دورا کی توریتامی تتین پتہ بائے کی تہ بستو کورا لکھلیک، زاپان کورا پیڅھیلیک وا تہ بوٹ کورا بیلیک۔ دورو کورمان کوری اچی تان کتابان، کاپیان، قلم پنسلان ای دفعہ لوڑی تان بستو لکھور، ہیہ جو تروئے منٹو کوروم۔
چھوچھی روپھی ہاسی بیتی اسکولوتین راہی کی ہو تامی، اسکولو سوم متعلقہ ہر ای اشناری تان ژاغا تیار بونی۔ بو اسقانیو سور ا  ہیتان لین بوئے وا ہے ہائی دیلیک نوبوئے۔ شیر استاذو ہمی لوان سورا غور اریر۔ ہمی لو ہوتین جم سارئے ہیتان سورا  عمل کوری ہنیسے بو آرامو سورا ،وختہ سکولہ توراؤ اوشوئے وا بو خوشان دی اوشوئے۔
ہنیسے شیر تان بچین ای اصول ساؤزئے اسیتائے۔ بوٹھان نیژی ای خاص ژاغا  وا تان بستو کمرائی التی میزا لکھوئے، اسکولار گیتی اسکولو زاپان پیڅھی ݰئیلی کوری الماری اوندیر۔ اچی ہوستان نیگی ݰپیک ژیبوئے۔ پھوک پوری ایھہ روپھی اسکولو کورمان کوروئے۔ اچی چھوچھو بچین کتابان، کاپیان وا قلم پنسلان لوڑی بستو صحیح کوری لکھوئے۔
 نمازدیگرار اچی تان ملگریان سوم ای بیتی اشٹوک کوروئے، وا شامو اذاناری پروشٹی اچی تان دورا تارور۔ دورا کا مینو کی ہانی مینوانتین اوغ الیک وا  ݰپیکار پروشٹی ہیتان ہوستان نیگئیک دی ہیس تنتین  ذمہ واری جوݰی اسیتائے وا ہے بچین کوس ہو بندئیکو ضرورت نو اوشوئے۔
شیرو دورو روئے ہو لوکار کوریکو، اسکولوتین شوقو سورا بیکو، جم محنت کوریکو وا تان اسکولو اشناریان خیالو لکھیکو وجہین ہورو ساری ہمیشہ خوشان اسیتانی۔ ہو ہمی جام عاداتان دیہو ہر ای نفری مثالو دیاؤ اوشونی۔ ہمی سف  جم عادت ہو تان استاذو دیرو ہدایاتان سورا عمل کوری  ہرکورمو ترتیبو اوچے وختہ کوریکو سورا پیدا بیتی اوشوئے۔استاذو ہیہ لو ہمیشہ ہو یادی بہچاؤ اوشوئے کی "وختو پابندی کیہ بولوٹ  دونیکو بݰ اشناری نو، بلکہ ہیس بو پھوک کورمہ شروع بوئے وا بو فائدہ تاریر۔"ہیہ وجہ اوشوئے کی شیر ہنیسے تان ہر ای کورمو ترتیبو سورا کوری زندگیہ تنتین آسانی پیدا کوری اسیتائے۔
 پیش کش: #مئیر ݮھترار  کھوار سفر نامہ: شمالی افریقو کوسیکو قصہ
جوؤ قسط: المھدیو سئیل
نیویشاک: احمد فراز

نستو قسطہ اسپہ تیونسو بارا مشقول بیتی اسیتم۔ ہنون تیونسار تقریباً تروئے گھنٹو پونہ ای تاریخی شہر کی، ہو نام المھدیہ شیر، ہو بارا مشقول بوسی۔ ادینہ انوس پیشینو تروئے بجا اڈو ویݰکی بغاتم۔ کورا کی ہل بوسیان ہموغار اڈوت گاڑیہ پھت گھنٹو پون شیر۔ ٹیکسی گنیتام گنی ہاتی نیشی ڈرائیورو سُم مشقول بیتی توتین لو پھریتام کی اسپہ مہدیو ویݰکی بیسیان۔ ڈرائیور خوشان ہوئے بیتی ریر کی اوا دی مہدیار مگم ہل ہیہ تیونسہ بومان۔ ہݰ بیتی اڈا توریکو ڈرائیور میکی اسپت ریر کی پسہ ہیا تان ہل بور اوا اڈائی بی پستے گاڑی برابر کوری اچی گوم۔ اسپہ ای پونچ منٹ انتظار اریتام ہانیہ اسپت گاڑی تیار اریر۔ گاڑی ہوباہو اسپہ ݯھترارو فالائنکوچ ! مگم جم ہݰ کی سیکنڈ سیٹو تروئے باقی جُوگان نیشئیران۔ اسپہ اوشٹ نفری ای فالائنکوچہ پورا ہوتم۔
مختصر ہݰ کی بوتو سوت اوچے پھتہ المھدیہ توریتام۔ ہیہ شہرو نامہ تان ای لوٹ تاریخ کھوشت شیر۔ فاطمی خلافتو بنیاد ہیا شروع بیتی شیر وا فاطمی خلیفہ عبداللہ المہدی ہیہ شہرو میربوختو لکھی اسور۔ چھوچھی آرامہ روپھی ݰپیک ژوتی کوساؤ بغاتم۔ بیشیر منٹ کوسی ای ہݰ ژاغہ توریتام کی ہتو لوڑی ہݰ ساریتائے گویا کی واقعی ہیا کیہ لوٹ سلطنتو آغاز بیتی شیر۔ ای لوٹ نوغورو دواختو غون دواخت مگم  بند نو بلکہ ہے موژین راہ بیتی پھائے نسہ پھائے نیسی شیر۔ ہیہ دواخت فاطمی دورا ساؤزونو بیتی شیر ہو نام باب زویلہ بیرائے۔

ہموغار ہتے وختو پرانو مہدیو آغاز بویان۔ ہوسُم ݯوکی تان ای عجائب گھر دی شیر ہیرا بھی بیشیر دینار دیتی اندرینی بین بویان۔ ہیہ میوزیمو اندرینی بیکار اچی ای راہ ہے لوٹ دواختوت بیران ہتیرا بیکو پورا شہرو پوشین بویان۔ میوزیمو اندرینی بو بو نوغ اشناریان سورا غیچ توریتائے ہے موژی ای فاطمی دورو سکہ دی غیچی ہائے۔

ہموغار ای پونچ منٹو پونہ ای لوٹ مسجد شیر۔ ہو نام “مسجد المہدیہ”۔ ہمو بارا ہݰ لو دونیان کی ہیہ فاطمی دورا ساؤزونو بیتی شیر۔ تقریباً 916ء ہیاکوروم شروع بیتی شیر وا ای ݯھوئے سال اچہ ہیہ مکمل بیتی المھدیو جامع مسجد ساؤز بیتی شیر۔ مسجدو پرانو وا مضبوط کھانج ہتے سُم جوستہ لوٹ لوٹ دروازہ وا کھولاؤ لوٹ شران دی شیر۔ مسجدو ناسین مین روڈ بیران وا روڈار موڑی لوٹ سمندر شیر۔ ہے راہو گنی کی بغاؤ بو ݰئیلی منظر تہ غیچی گونی مگم پھوک دودیری کی بغاؤ ای لوٹ قبرستان شیر۔ راہو توری موڑی ویݰکی غیچ الیپھیکہ پت قبر تان تہ غیچی گونی۔ مہ پوشیرو سفان سار لوٹ قبرستان ہیہ اوشوئے۔

ہمو تاریخ دی بی البت فاطمی دورو سے سوم تارور۔ مگم کیہ تصدیق دیکو  نو  بوم۔ ہیغین ہݰ ریم کی ہیہ بارا مزید تحقیقو ضرورت شیر

پیش کش: #مئیرݯھترار
"""

def compute_perplexity_text(text, block_size=1024):
    encodings = tokenizer(text, return_tensors="pt")
    input_ids = encodings.input_ids.to(model.device)

    total_loss = 0.0
    total_tokens = 0
    model.eval()
    with torch.no_grad():
        for i in range(0, input_ids.size(1), block_size):
            chunk = input_ids[:, i:i+block_size]
            if chunk.size(1) < 2:
                continue
            outputs = model(chunk, labels=chunk)
            loss = outputs.loss.item()
            n_tokens = chunk.size(1)
            total_loss += loss * n_tokens
            total_tokens += n_tokens

    avg_loss = total_loss / total_tokens
    ppl = math.exp(avg_loss)
    return avg_loss, ppl, total_tokens

avg_loss, ppl, n_tok = compute_perplexity_text(fresh_text)
print(f"Tokens evaluated: {n_tok}")
print(f"Average loss: {avg_loss:.4f}")
print(f"Perplexity: {ppl:.2f}")

Tokens evaluated: 4647
Average loss: 2.4348
Perplexity: 11.41
